# Tópicos de Estadística Avanzada - Clase 1
## Vectores Aleatorios y Distribuciones Multivariadas

**Dra. Beatriz Marrón - Dr. José Bavio**  
Universidad Nacional del Sur - 2026

---

## 1. Vectores Aleatorios

### Definición
Un **vector aleatorio** es una función $\mathbf{X}: \Omega \to \mathbb{R}^n$ tal que para cualquier conjunto $B \in \mathcal{B}(\mathbb{R}^n)$, se cumple que $\mathbf{X}^{-1}(B) \in \mathcal{A}$.

Se representa como $\mathbf{X} = (X_1, \ldots, X_n)$ donde cada $X_i$ es una variable aleatoria.

### Función de Distribución Conjunta
La función de distribución conjunta de un vector $(X, Y)$ es:
$$F(x, y) = P(X \leq x, Y \leq y)$$

### Función de Densidad Conjunta
Para un vector **absolutamente continuo**, existe $f(x,y) \geq 0$ tal que:
$$F(x, y) = \int_{-\infty}^{x} \int_{-\infty}^{y} f(u, v) \, dv \, du$$

**Propiedades:**
1. $f(x, y) \geq 0$
2. $\int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x, y) \, dx \, dy = 1$
3. $f(x, y) = \frac{\partial^2}{\partial x \partial y} F(x, y)$

### En criollo

Un **vector aleatorio** es simplemente un conjunto de variables aleatorias que viven juntas. Por ejemplo, si medís la altura y el peso de una persona, $(X, Y) = (\text{altura}, \text{peso})$ es un vector aleatorio de dimensión 2.

La **distribución conjunta** te dice la probabilidad de que ambas variables tomen ciertos valores *al mismo tiempo*. Es como preguntarte: ¿qué probabilidad hay de que alguien mida menos de 1.70m **Y** pese menos de 70kg?

La **densidad conjunta** $f(x,y)$ es como un "paisaje de probabilidad" en 3D: donde la función es más alta, hay más probabilidad de encontrar valores de $(X,Y)$ en esa región.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from mpl_toolkits.mplot3d import Axes3D

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Ejemplo: Normal Bivariada
mu_x, mu_y = 0, 0
sigma_x, sigma_y = 1, 1.5
rho = 0.7  # correlación

# Crear grilla
x = np.linspace(-4, 4, 100)
y = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(x, y)

# Matriz de covarianza
cov = [[sigma_x**2, rho * sigma_x * sigma_y],
       [rho * sigma_x * sigma_y, sigma_y**2]]

# Crear distribución normal bivariada
pos = np.dstack((X, Y))
rv = stats.multivariate_normal([mu_x, mu_y], cov)
Z = rv.pdf(pos)

# Visualización 3D
fig = plt.figure(figsize=(12, 5))

# Superficie 3D
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('f(x,y)')
ax1.set_title(f'Densidad Conjunta\nρ = {rho}')

# Curvas de nivel
ax2 = fig.add_subplot(122)
contour = ax2.contour(X, Y, Z, levels=10, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('Curvas de Nivel')
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

print(f"Parámetros de la distribución:")
print(f"  μ_X = {mu_x}, σ_X = {sigma_x}")
print(f"  μ_Y = {mu_y}, σ_Y = {sigma_y}")
print(f"  ρ = {rho}")

## 2. Distribuciones Marginales y Condicionales

### Densidades Marginales
La densidad marginal de $X$ se obtiene "integrando" sobre $Y$:
$$f_X(x) = \int_{-\infty}^{\infty} f(x, y) \, dy$$

Análogamente para $Y$:
$$f_Y(y) = \int_{-\infty}^{\infty} f(x, y) \, dx$$

### Densidad Condicional
Si $f_Y(y) \neq 0$:
$$f_{X|Y}(x|y) = \frac{f(x, y)}{f_Y(y)}$$

### Independencia
$X$ e $Y$ son **independientes** ($X \perp Y$) si y sólo si:
$$f(x, y) = f_X(x) \cdot f_Y(y), \quad \forall (x,y)$$

### En criollo

Las **marginales** son lo que pasa cuando "te olvidás" de una de las variables. Si tenés la distribución conjunta de altura y peso, la marginal de altura es simplemente la distribución de alturas ignorando el peso.

La **condicional** $f_{X|Y}(x|y)$ te dice: "si ya sé que $Y = y$, ¿cómo se distribuye $X$?" Por ejemplo: si ya sé que alguien pesa 80kg, ¿cómo se distribuye su altura?

La **independencia** significa que saber el valor de una variable no te da ninguna información sobre la otra. Si altura y peso fueran independientes (spoiler: no lo son), saber cuánto pesás no cambiaría la distribución de tu altura.

In [ ]:
# Generar muestra de la normal bivariada
np.random.seed(42)
n_samples = 1000
samples = rv.rvs(n_samples)

# Visualizar marginales y conjunta
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Scatter plot con histogramas marginales
ax_joint = axes[1, 0]
ax_marg_x = axes[0, 0]
ax_marg_y = axes[1, 1]
axes[0, 1].axis('off')

# Joint scatter
ax_joint.scatter(samples[:, 0], samples[:, 1], alpha=0.3, s=10)
ax_joint.set_xlabel('X')
ax_joint.set_ylabel('Y')
ax_joint.grid(True, alpha=0.3)

# Marginal X
ax_marg_x.hist(samples[:, 0], bins=30, density=True, alpha=0.6, color='blue', label='Empírica')
x_range = np.linspace(-4, 4, 100)
ax_marg_x.plot(x_range, stats.norm.pdf(x_range, mu_x, sigma_x), 'r-', lw=2, label='Teórica')
ax_marg_x.set_ylabel('Densidad')
ax_marg_x.legend()
ax_marg_x.set_title('Marginal de X')
ax_marg_x.grid(True, alpha=0.3)

# Marginal Y
ax_marg_y.hist(samples[:, 1], bins=30, density=True, alpha=0.6, color='green', 
               orientation='horizontal', label='Empírica')
y_range = np.linspace(-4, 4, 100)
ax_marg_y.plot(stats.norm.pdf(y_range, mu_y, sigma_y), y_range, 'r-', lw=2, label='Teórica')
ax_marg_y.set_xlabel('Densidad')
ax_marg_y.legend()
ax_marg_y.set_title('Marginal de Y')
ax_marg_y.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nEstadísticos muestrales:")
print(f"  Media muestral X: {np.mean(samples[:, 0]):.3f} (teórica: {mu_x})")
print(f"  Media muestral Y: {np.mean(samples[:, 1]):.3f} (teórica: {mu_y})")
print(f"  Desvío muestral X: {np.std(samples[:, 0], ddof=1):.3f} (teórico: {sigma_x})")
print(f"  Desvío muestral Y: {np.std(samples[:, 1], ddof=1):.3f} (teórico: {sigma_y})")

## 3. Características Numéricas

### Esperanza de un Vector
$$\mathbb{E}(\mathbf{X}) = (\mathbb{E}(X_1), \ldots, \mathbb{E}(X_n))$$

### Covarianza
$$\text{Cov}(X, Y) = \mathbb{E}[(X - \mathbb{E}(X))(Y - \mathbb{E}(Y))] = \mathbb{E}(XY) - \mathbb{E}(X)\mathbb{E}(Y)$$

**Propiedades:**
1. $\text{Cov}(X, Y) = \text{Cov}(Y, X)$
2. $\text{Cov}(X, X) = \text{Var}(X)$
3. Si $X \perp Y$ entonces $\text{Cov}(X, Y) = 0$ (pero no al revés!)

### Coeficiente de Correlación
$$\rho(X, Y) = \frac{\text{Cov}(X, Y)}{\sqrt{\text{Var}(X) \cdot \text{Var}(Y)}}$$

**Propiedades:**
1. $-1 \leq \rho(X, Y) \leq 1$
2. $|\rho(X, Y)| = 1$ si y sólo si existe relación lineal exacta: $Y = aX + b$

### Matriz de Covarianza
$$\Sigma = \begin{pmatrix}
\text{Var}(X_1) & \text{Cov}(X_1, X_2) & \cdots & \text{Cov}(X_1, X_n) \\
\text{Cov}(X_2, X_1) & \text{Var}(X_2) & \cdots & \text{Cov}(X_2, X_n) \\
\vdots & \vdots & \ddots & \vdots \\
\text{Cov}(X_n, X_1) & \text{Cov}(X_n, X_2) & \cdots & \text{Var}(X_n)
\end{pmatrix}$$

### En criollo

La **covarianza** mide si dos variables "se mueven juntas". Si cuando una sube la otra también tiende a subir, la covarianza es positiva. Si una sube y la otra baja, es negativa. Si no hay relación, es cercana a cero.

El problema de la covarianza es que depende de las unidades. Si medís la altura en metros o en centímetros, la covarianza cambia. Por eso se usa la **correlación** $\rho$, que está siempre entre -1 y 1:
- $\rho = 1$: relación lineal perfecta positiva (los puntos caen en una recta con pendiente positiva)
- $\rho = -1$: relación lineal perfecta negativa
- $\rho = 0$: no hay relación lineal (¡pero puede haber otras relaciones!)

La **matriz de covarianza** $\Sigma$ es simétrica y contiene todas las varianzas en la diagonal y todas las covarianzas fuera de la diagonal.

In [ ]:
# Visualizar el efecto de diferentes correlaciones
correlaciones = [-0.9, -0.5, 0, 0.5, 0.9]
fig, axes = plt.subplots(1, 5, figsize=(16, 3))

for i, rho_i in enumerate(correlaciones):
    # Matriz de covarianza
    cov_i = [[1, rho_i], [rho_i, 1]]
    rv_i = stats.multivariate_normal([0, 0], cov_i)
    samples_i = rv_i.rvs(300)
    
    # Calcular correlación muestral
    rho_sample = np.corrcoef(samples_i[:, 0], samples_i[:, 1])[0, 1]
    
    axes[i].scatter(samples_i[:, 0], samples_i[:, 1], alpha=0.5, s=15)
    axes[i].set_xlabel('X')
    if i == 0:
        axes[i].set_ylabel('Y')
    axes[i].set_title(f'ρ = {rho_i}\n(muestra: {rho_sample:.2f})')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(-3, 3)
    axes[i].set_ylim(-3, 3)
    axes[i].set_aspect('equal')

plt.tight_layout()
plt.show()

# Calcular matriz de covarianza de la muestra original
cov_sample = np.cov(samples.T)
corr_sample = np.corrcoef(samples.T)

print("\nMatriz de Covarianza (muestral):")
print(cov_sample)
print("\nMatriz de Correlación (muestral):")
print(corr_sample)
print(f"\nCorrelación teórica: {rho:.3f}")
print(f"Correlación muestral: {corr_sample[0, 1]:.3f}")

## 4. Distribución Normal Bivariada

### Definición
Un vector $(X, Y)$ tiene distribución **Normal Bivariada** con parámetros $(\mu_x, \mu_y, \sigma_x^2, \sigma_y^2, \rho)$ si su densidad es:

$$f(x, y) = \frac{1}{2\pi\sigma_x\sigma_y\sqrt{1-\rho^2}} \exp\left\{-\frac{1}{2(1-\rho^2)}\left[\left(\frac{x-\mu_x}{\sigma_x}\right)^2 - 2\rho\left(\frac{x-\mu_x}{\sigma_x}\right)\left(\frac{y-\mu_y}{\sigma_y}\right) + \left(\frac{y-\mu_y}{\sigma_y}\right)^2\right]\right\}$$

### Propiedades Importantes

1. **Marginales**: Ambas son normales
   - $X \sim N(\mu_x, \sigma_x^2)$
   - $Y \sim N(\mu_y, \sigma_y^2)$

2. **Condicionales**: También son normales
   - $X|Y=y \sim N\left(\mu_x + \rho\frac{\sigma_x}{\sigma_y}(y - \mu_y), \sigma_x^2(1-\rho^2)\right)$
   - $Y|X=x \sim N\left(\mu_y + \rho\frac{\sigma_y}{\sigma_x}(x - \mu_x), \sigma_y^2(1-\rho^2)\right)$

3. **Independencia**: $X \perp Y$ si y sólo si $\rho = 0$

### En criollo

La **Normal Bivariada** es la distribución multivariada más importante en estadística. Es como la "campana de Gauss" pero en 2D: en lugar de una campana, tenés una "montaña" de probabilidad.

Lo copado es que:
- Si mirás solo una variable (marginal), es normal
- Si fijás una variable y mirás la otra (condicional), también es normal
- La "forma" de la montaña está completamente determinada por 5 parámetros: las dos medias, las dos varianzas, y la correlación

**Caso especial**: Para la normal bivariada (y sólo para ella), correlación cero implica independencia. En general esto no es cierto para otras distribuciones.

In [ ]:
# Visualizar distribuciones condicionales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Parámetros
mu_x, mu_y = 0, 0
sigma_x, sigma_y = 1, 1.5
rho = 0.7

# Valores fijos para condicionar
y_fixed_values = [-2, 0, 2]
colors = ['blue', 'red', 'green']

# Crear grilla para X
x_range = np.linspace(-4, 4, 200)

for ax_idx, (ax, y_fix) in enumerate(zip(axes, y_fixed_values)):
    # Parámetros de la condicional X|Y=y_fix
    mu_cond = mu_x + rho * (sigma_x / sigma_y) * (y_fix - mu_y)
    sigma_cond = sigma_x * np.sqrt(1 - rho**2)
    
    # Densidad condicional
    f_cond = stats.norm.pdf(x_range, mu_cond, sigma_cond)
    
    ax.plot(x_range, f_cond, lw=2, label=f'$f_{{X|Y}}(x|y={y_fix})$')
    ax.axvline(mu_cond, color='red', linestyle='--', alpha=0.7, label=f'$\\mu_{{X|Y={y_fix}}}={mu_cond:.2f}$')
    ax.fill_between(x_range, 0, f_cond, alpha=0.2)
    ax.set_xlabel('x')
    ax.set_ylabel('Densidad')
    ax.set_title(f'Condicional X dado Y={y_fix}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Distribuciones Condicionales:")
for y_fix in y_fixed_values:
    mu_cond = mu_x + rho * (sigma_x / sigma_y) * (y_fix - mu_y)
    sigma_cond = sigma_x * np.sqrt(1 - rho**2)
    print(f"  X|Y={y_fix:+.0f} ~ N({mu_cond:.3f}, {sigma_cond**2:.3f})")

# Mostrar que la varianza condicional NO depende de y
sigma_cond = sigma_x * np.sqrt(1 - rho**2)
print(f"\nNota: σ²(X|Y) = {sigma_cond**2:.3f} es constante (no depende del valor de Y)")

## 5. Funciones Generadoras (breve introducción)

### Función Generadora de Momentos (f.g.m.)
$$M(t) = \mathbb{E}(e^{tX})$$

**Propiedad clave**: $M^{(n)}(0) = \mathbb{E}(X^n)$

### Función Característica
$$\phi(t) = \mathbb{E}(e^{itX})$$

**Ventaja**: Siempre existe (la f.g.m. puede no existir)

**Propiedad fundamental**: $\phi(t)$ determina **de manera única** a la distribución $F(x)$

### En criollo

Las **funciones generadoras** son transformaciones matemáticas de la distribución. Son útiles porque:

1. **Simplifican cálculos**: En lugar de trabajar con integrales complicadas, trabajás con funciones más manejables
2. **Identifican distribuciones**: Dos distribuciones distintas tienen funciones características distintas
3. **Generan momentos**: Derivando la f.g.m. en cero, obtenés todos los momentos

La **función característica** es como la transformada de Fourier de la densidad. Siempre existe (a diferencia de la f.g.m.) y tiene buenas propiedades para trabajar con sumas de variables aleatorias independientes.

In [ ]:
# Ejemplo: función característica de una Normal(0,1)
# φ(t) = exp(-t²/2)

t_values = np.linspace(-3, 3, 200)

# Función característica teórica
phi_real = np.exp(-t_values**2 / 2)  # Parte real
phi_imag = np.zeros_like(t_values)    # Parte imaginaria (es cero para Normal)

# Estimación empírica de la función característica
np.random.seed(42)
X_samples = np.random.standard_normal(10000)

phi_emp_real = np.zeros_like(t_values)
phi_emp_imag = np.zeros_like(t_values)

for i, t in enumerate(t_values):
    exp_itX = np.exp(1j * t * X_samples)
    phi_emp_real[i] = np.real(exp_itX).mean()
    phi_emp_imag[i] = np.imag(exp_itX).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Parte real
axes[0].plot(t_values, phi_real, 'r-', lw=2, label='Teórica')
axes[0].plot(t_values, phi_emp_real, 'b.', alpha=0.5, markersize=3, label='Empírica')
axes[0].set_xlabel('t')
axes[0].set_ylabel('Re(φ(t))')
axes[0].set_title('Parte Real de la Función Característica\nN(0,1)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parte imaginaria
axes[1].plot(t_values, phi_imag, 'r-', lw=2, label='Teórica')
axes[1].plot(t_values, phi_emp_imag, 'b.', alpha=0.5, markersize=3, label='Empírica')
axes[1].set_xlabel('t')
axes[1].set_ylabel('Im(φ(t))')
axes[1].set_title('Parte Imaginaria de la Función Característica\nN(0,1)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Función característica de N(0,1): φ(t) = exp(-t²/2)")
print("\nPrimeros momentos usando la f.g.m.:")
print(f"  E(X) = M'(0) = 0")
print(f"  E(X²) = M''(0) = 1")
print(f"  Var(X) = E(X²) - [E(X)]² = 1 - 0 = 1")

---
## Preguntas para Responder

### Pregunta 1: Explorando la Correlación (requiere modificar código)

En la sección donde graficamos diferentes valores de correlación, modificá el código para:

a) Generar una Normal Bivariada con $\rho = 0$ (variables independientes)

b) Crear manualmente dos variables $U$ y $V$ donde:
   - $U \sim N(0, 1)$
   - $V = U^2$

c) Calculá la correlación entre $U$ y $V$

d) Hacé un scatter plot de $(U, V)$

**Pregunta**: ¿Son $U$ y $V$ independientes? ¿Cuál es su correlación? ¿Qué conclusión sacás sobre la relación entre correlación cero e independencia?

---

### Pregunta 2: Interpretando la Distribución Condicional (teórica/interpretativa)

Mirá los gráficos de las distribuciones condicionales $f_{X|Y}(x|y)$ para diferentes valores de $y$.

a) ¿Qué pasa con la media de la distribución condicional cuando $y$ aumenta? ¿Por qué?

b) ¿Qué pasa con la varianza de la distribución condicional cuando cambiás $y$? ¿Por qué?

c) Si $\rho = 0$, ¿cómo serían estas distribuciones condicionales?

d) En el contexto de altura y peso: Si la correlación es positiva y sabés que alguien es muy alto, ¿esperarías que su peso esté por encima o por debajo del promedio? Justificá con la fórmula de la distribución condicional.

---

### Pregunta 3: Marginales vs Conjunta (requiere modificar código)

Considerá dos distribuciones:

**Distribución A**: Normal Bivariada con $\mu_X = 0$, $\mu_Y = 0$, $\sigma_X = 1$, $\sigma_Y = 1$, $\rho = 0.8$

**Distribución B**: Normal Bivariada con $\mu_X = 0$, $\mu_Y = 0$, $\sigma_X = 1$, $\sigma_Y = 1$, $\rho = -0.8$

a) Generá muestras de ambas distribuciones y graficá los scatter plots

b) Calculá y graficá las distribuciones marginales de $X$ e $Y$ para ambos casos

**Pregunta**: ¿Las marginales de A y B son iguales? ¿Las conjuntas son iguales? ¿Qué te dice esto sobre la información que perdés al mirar solo las marginales?